# Caching

**What this notebook does**

* Builds a filtered, enriched DataFrame of Boston customers (`df_base`) and caches it.
* Derives two DataFrames (`df1`, `df2`) from the cached base and inspects their plans to see the cache being reused.
* Re-persists the base with an explicit `StorageLevel` and explains the storage level options.

**Core concept: caching**

Spark DataFrames are lazy: every action recomputes the full lineage from the source files. `cache()` / `persist()` tell Spark to keep the computed result in executor memory (and/or disk) after the first action, so later actions on the same DataFrame, or on DataFrames derived from it, read from the cache (`InMemoryRelation` / `InMemoryTableScan` in the plan) instead of re-reading and re-transforming the source.

**Databricks conversion notes**

* The local Jupyter session setup (`SparkSession.builder`, `sparkContext`, `spark.driver.memory`, Kryo serializer) was removed. Databricks provides `spark`; driver memory and the serializer are cluster-level settings (compute > Advanced > Spark config) and cannot be changed from a notebook.
* The local `../data/data_skew` path was replaced with the Unity Catalog volume `/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed`.
* Standard access mode and serverless use Spark Connect: a DataFrame cannot be used after `unpersist()`, so the notebook rebuilds `df_base` before re-persisting it.
* Databricks also has an automatic **disk cache** for Parquet/Delta files on classic compute, separate from Spark's `cache()`; it speeds up repeated file reads without any code.
* Run on a classic cluster (e.g. Sandani Personal Compute) to see cache operators and the **Storage** tab in the Spark UI; serverless manages caching for you and exposes less of it.

## 1. Setup

In [0]:
import warnings
warnings.filterwarnings("ignore")

In [0]:
# Jupyter-only InteractiveShell setting removed; Databricks renders cell output natively.

# Source data in the Unity Catalog volume
base_path = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed"

In [0]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is not imported: Databricks provides a ready-to-use `spark` object.

In [0]:
# Original (local Jupyter):
#   spark = (SparkSession.builder
#            .config("spark.driver.memory", "10g")
#            .config("spark.sql.files.maxPartitionBytes", "268435456")
#            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
#            .master("local[*]").getOrCreate())
#   sc = spark.sparkContext; sc.setLogLevel("ERROR")
# driver.memory and serializer are cluster-level settings on Databricks; only the session-level
# maxPartitionBytes (256 MB read partitions) can be set here.
try:
    spark.conf.set("spark.sql.files.maxPartitionBytes", "268435456")
    print("maxPartitionBytes =", spark.conf.get("spark.sql.files.maxPartitionBytes"))
except Exception as e:
    print(f"Could not set maxPartitionBytes on this compute: {type(e).__name__}")

In [0]:
customers_file = f"{base_path}/customers.parquet"
df_customers = spark.read.parquet(customers_file)

In [0]:
display(df_customers.limit(5))

## 2. Caching a Reused DataFrame

**What:** Filter to Boston customers, add an age-based `customer_group`, and cache the result as `df_base`.

**Core concept:** `cache()` is **lazy**: it only marks the DataFrame. The first action (`count()`) computes it and stores the partitions; later actions reuse them. Cache a DataFrame when it is reused several times and is expensive to recompute; otherwise caching just costs memory.

In [0]:
def build_base():
    """Boston customers with an age-based customer_group (rebuilt after unpersist under Spark Connect)."""
    return (
        df_customers
        .filter(F.col("city") == "boston")
        .withColumn(
            "customer_group",
            F.when(F.col("age").between(20, 30), F.lit("young"))
            .when(F.col("age").between(31, 50), F.lit("mid"))
            .when(F.col("age") > 51, F.lit("old"))
            .otherwise(F.lit("kid"))
        )
        .select("cust_id", "name", "age", "gender", "birthday", "zip", "city", "customer_group")
    )

df_base = build_base()
df_base.cache()      # lazy: only marks the DataFrame for caching
df_base.count()      # action: computes df_base once and fills the cache
display(df_base.limit(5))

**What:** Build `df1` and `df2` on top of the cached `df_base` and print their full plans with `explain(True)`.

**Core concept:** Look at the **Physical Plan**. Instead of `FileScan parquet` + `Filter`, it should start from `InMemoryTableScan` over an `InMemoryRelation`: Spark recognises that part of the plan is already cached and reads it from memory. Only the new columns (`test_column_*`, `birth_year` / `birth_month`) are computed.

In [0]:
df1 = (
    df_base
    .withColumn("test_column_1", F.lit("test_column_1"))
    .withColumn("birth_year", F.split("birthday", "/").getItem(2))
)

df1.explain(True)
display(df1.limit(5))

In [0]:
df2 = (
    df_base
    .withColumn("test_column_2", F.lit("test_column_2"))
    .withColumn("birth_month", F.split("birthday", "/").getItem(1))
)

df2.explain(True)
display(df2.limit(5))

## 3. `StorageLevel` Types

(As of Spark `3.4`)

- `DISK_ONLY`: CPU efficient, memory efficient, slow to access, data is serialized when stored on disk
- `DISK_ONLY_2`: disk only, replicated 2x
- `DISK_ONLY_3`: disk only, replicated 3x

- `MEMORY_AND_DISK`: spills to disk if there's no space in memory
- `MEMORY_AND_DISK_2`: memory and disk, replicated 2x
- `MEMORY_AND_DISK_DESER`(default): same as `MEMORY_AND_DISK`, deserialized in both for fast access

- `MEMORY_ONLY`: CPU efficient, memory intensive
- `MEMORY_ONLY_2`: memory only, replicated 2x - for resilience, if one executor fails

**Note**: 
- `SER` is CPU intensive, memory saving as data is compact while `DESER` is CPU efficient, memory intensive
- Size of data on disk is lesser as data is in serialized format, while deserialized in memory as JVM objects for faster access

### When to use what?
```
Storage Level    Space used  CPU time  In memory  On-disk  Serialized
---------------------------------------------------------------------
MEMORY_ONLY          High        Low       Y          N        N         
MEMORY_ONLY_SER      Low         High      Y          N        Y     
MEMORY_AND_DISK      High        Medium    Some       Some     Some  
MEMORY_AND_DISK_SER  Low         High      Some       Some     Y     
DISK_ONLY            Low         High      N          Y        Y     
```

**Databricks / PySpark notes**

* `cache()` on a DataFrame uses `MEMORY_AND_DISK_DESER` by default.
* The `_SER` levels in the table are Scala/Java options; PySpark exposes only the levels listed above (Python data is always stored serialized).
* Replicated levels (`_2`, `_3`) only add value on multi-node clusters; on a single-node cluster there is no second executor to hold the copy.
* On classic compute, check what is cached and how much memory it uses in **Spark UI > Storage**.

**What:** Drop the existing cache and persist `df_base` again with `StorageLevel.MEMORY_ONLY`.

**Core concept:** A DataFrame's storage level cannot be changed while it is cached, so `unpersist()` comes first. With `MEMORY_ONLY`, partitions that do not fit in memory are not cached and are recomputed from the source when needed, instead of spilling to disk as with `MEMORY_AND_DISK`.

In [0]:
# Remove the existing cache, then persist with an explicit storage level.
df_base.unpersist()

# Spark Connect (standard access mode / serverless): a DataFrame cannot be reused after unpersist
# (CACHE_DATAFRAME_ACCESS_AFTER_UNCACHE), so rebuild it before persisting again.
df_base = build_base().persist(StorageLevel.MEMORY_ONLY)
df_base.count()   # action: materialize the new cache
print("Storage level:", df_base.storageLevel)

# Derived DataFrames must also be rebuilt from the new df_base
df1 = (
    df_base
    .withColumn("test_column_1", F.lit("test_column_1"))
    .withColumn("birth_year", F.split("birthday", "/").getItem(2))
)

display(df1.limit(5))

In [0]:
# Original: spark.stop()  -> not used on Databricks; the session is managed by the compute.
# Release the cached data instead.
df_base.unpersist()